# 🍀 CLOVES: Interactive Quickstart & Benchmark Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huynhldvn/CLOVES/blob/master/notebooks/quickstart_interactive_demo.ipynb)
[![PyPI version](https://img.shields.io/pypi/v/cloves.svg)](https://pypi.org/project/cloves/)

This clean, self-contained notebook lets you test **CLOVES (Cleft-Overstep Optimization)** immediately with zero configuration.

### What this notebook demonstrates:
1. **2D Ill-Conditioned Ravine Simulation**: Visualizes why AdamW and LION oscillate across canyon walls while CLOVES descends smoothly down the floor.
2. **Standard Vision Training**: 2-line setup using `CLOVES_Perpendicular` on a neural classification network.
3. **LLM LoRA Tuning**: 2-line setup using `CLOVES_Apex_Asymmetric` with asymmetric LoRA A/B velocity scaling.

## 1. Installation & Environment Check

In [ ]:
try:
    import cloves
except ImportError:
    !pip install -q cloves
    import cloves

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from cloves import (
    CLOVES,
    CLOVES_Bisector,
    CLOVES_Perpendicular,
    CLOVES_Affine,
    CLOVES_Adaptive,
    CLOVES_LLM,
    CLOVES_Apex_Asymmetric,
    CLOVES_Matrix_Polar,
)

print(f"Successfully imported CLOVES v{cloves.__version__}!")
print(f"CUDA Available: {torch.cuda.is_available()}")

## 2. Benchmark 1: 2D Ill-Conditioned Ravine (Condition Number $\kappa = 100$)

We test on the anisotropic quadratic ravine:
$$f(x, y) = 0.5 (x^2 + 100 y^2), \quad \text{starting at } (x_0, y_0) = (10.0, 1.0)$$

The valley walls are 100x steeper than the valley floor. Let's compare **CLOVES-Perpendicular** against **AdamW** and **SGD-M**.

In [ ]:
def run_trajectory(opt_cls, opt_kwargs, steps=250):
    point = torch.tensor([10.0, 1.0], requires_grad=True)
    opt = opt_cls([point], **opt_kwargs)
    history = [point.detach().clone().numpy()]
    
    for _ in range(steps):
        opt.zero_grad()
        loss = 0.5 * (point[0]**2 + 100.0 * point[1]**2)
        loss.backward()
        opt.step()
        history.append(point.detach().clone().numpy())
        
    return np.array(history), loss.item()

# Run simulations
traj_cloves, loss_cloves = run_trajectory(CLOVES_Perpendicular, {"lr": 0.02, "q": 1.6, "gamma": 0.10, "gamma_perp": 0.25})
traj_adamw, loss_adamw = run_trajectory(torch.optim.AdamW, {"lr": 0.02, "betas": (0.9, 0.999)})
traj_sgdm, loss_sgdm = run_trajectory(torch.optim.SGD, {"lr": 0.001, "momentum": 0.9})

print("=" * 50)
print(f"CLOVES Final Loss: {loss_cloves:.6e} | Final Pos: x={traj_cloves[-1, 0]:.4f}, y={traj_cloves[-1, 1]:.2e}")
print(f"AdamW  Final Loss: {loss_adamw:.6f}   | Final Pos: x={traj_adamw[-1, 0]:.4f}, y={traj_adamw[-1, 1]:.2e}")
print(f"SGD-M  Final Loss: {loss_sgdm:.6f}   | Final Pos: x={traj_sgdm[-1, 0]:.4f}, y={traj_sgdm[-1, 1]:.2e}")
print("=" * 50)

In [ ]:
# Plot the optimization trajectories
plt.figure(figsize=(10, 5), dpi=120)
plt.plot(traj_adamw[:, 0], traj_adamw[:, 1], color="#ef4444", linestyle="--", alpha=0.7, label=f"AdamW (Loss: {loss_adamw:.2f})")
plt.plot(traj_sgdm[:, 0], traj_sgdm[:, 1], color="#6b7280", linestyle=":", label=f"SGD-M (Loss: {loss_sgdm:.2f})")
plt.plot(traj_cloves[:, 0], traj_cloves[:, 1], color="#10b981", linewidth=2.5, label=f"CLOVES-Perpendicular (Loss: {loss_cloves:.2e})")

plt.scatter([0], [0], color="#047857", s=100, marker="*", label="Global Optimum (0, 0)")
plt.scatter([10.0], [1.0], color="#000000", s=50, label="Start (10, 1)")

plt.title("Optimization Trajectories on 2D Ravine (Condition Number $\\kappa=1000$)", fontsize=13, fontweight="bold")
plt.xlabel("Longitudinal Coordinate x (Valley Floor)")
plt.ylabel("Transversal Coordinate y (Steep Walls)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(frameon=True, facecolor="white", framealpha=0.9)
plt.tight_layout()
plt.show()

## 3. Demo: Training a Neural Classification Network with CLOVES_Perpendicular

For standard vision / CNN / classification workloads, use `CLOVES_Perpendicular` as a drop-in replacement for AdamW or SGD-M.

In [ ]:
model = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

# 1-State Memory Optimizer (4 bytes / parameter)
optimizer = CLOVES_Perpendicular(model.parameters(), lr=0.05, beta=0.9, q=1.5, gamma=0.10)
criterion = nn.CrossEntropyLoss()

# Fixed training batch to demonstrate steady loss reduction
torch.manual_seed(42)
inputs = torch.randn(32, 64)
targets = torch.randint(0, 10, (32,))

# Run 10 training steps
for step in range(1, 11):
    optimizer.zero_grad()
    loss = criterion(model(inputs), targets)
    loss.backward()
    optimizer.step()
    
    if step % 2 == 0 or step == 1:
        print(f"Step {step:2d}/10 | Loss: {loss.item():.4f}")

print("Classification network trained successfully!")


## 4. Demo: Parameter-Efficient LoRA Tuning with CLOVES_Apex_Asymmetric

For LLMs and Transformers using LoRA, `CLOVES_Apex_Asymmetric` automatically identifies LoRA Matrix A ($r \times d_{in}$, Gaussian init) vs Matrix B ($d_{out} \times r$, Zero init) and assigns asymmetric overstep acceleration.

In [ ]:
class LoRAAttentionBlock(nn.Module):
    def __init__(self, dim=256, rank=16):
        super().__init__()
        # Standard scaled initialization so variance is 1.0
        self.frozen_weight = nn.Parameter(torch.randn(dim, dim) / (dim ** 0.5), requires_grad=False)
        # Matrix A (Gaussian init, r x dim)
        self.lora_A = nn.Parameter(torch.randn(rank, dim) * 0.05)
        # Matrix B (Zero init, dim x r)
        self.lora_B = nn.Parameter(torch.zeros(dim, rank))
        
    def forward(self, x):
        base = x @ self.frozen_weight.T
        adapter = (x @ self.lora_A.T) @ self.lora_B.T
        return base + adapter

torch.manual_seed(42)
lora_model = LoRAAttentionBlock()

# Dedicated LoRA Asymmetric Optimizer
lora_optimizer = CLOVES_Apex_Asymmetric(
    [p for p in lora_model.parameters() if p.requires_grad],
    lr=1e-3,
    total_steps=10
)

# Fixed instruction adaptation batch to demonstrate adapter convergence
X_train = torch.randn(16, 256)
target = lora_model(X_train).detach() + 0.3 * torch.randn(16, 256)

for step in range(1, 11):
    lora_optimizer.zero_grad()
    pred = lora_model(X_train)
    loss = nn.functional.mse_loss(pred, target)
    loss.backward()
    lora_optimizer.step()
    
    if step % 2 == 0 or step == 1:
        print(f"LoRA Step {step:2d}/10 | Loss: {loss.item():.4f}")

print("LoRA fine-tuning step completed successfully!")
